# Project 1 Full Workflow Code Notebook

Canonical one-file workflow for grading under the one-code-file assumption.

This notebook integrates: acquisition checks, cleaning/preprocessing, EDA, feature engineering, diagnostics, and ablation.


## 1) Imports and Configuration

In [1]:
from __future__ import annotations

import json
import re
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gensim import corpora
from gensim.models import LdaModel
from scipy.stats import chi2_contingency, kruskal, spearmanr
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
REQUIRED_RAW_COLUMNS = ["title", "body", "url", "score", "comms_num", "timestamp"]

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'Code Files' else Path.cwd().resolve()
if not (ROOT / 'Project Deliverables').exists():
    ROOT = Path('/Users/m2/Projects/STAT 5243/Project 1')

RAW_PATH = ROOT / 'Project Deliverables' / 'Datasets' / 'reddit_wsb.csv'
CLEANED_CANONICAL_PATH = ROOT / 'Project Deliverables' / 'Datasets' / 'reddit_wsb_cleaned.csv'
OUT_DIR = ROOT / 'Project Workspace' / 'Supporting Materials' / 'Generated Outputs' / 'one_code_notebook'
FIG_DIR = OUT_DIR / 'artifacts' / 'figures'
JSON_DIR = OUT_DIR / 'artifacts' / 'json'

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('RAW_PATH exists:', RAW_PATH.exists())
print('CANONICAL CLEANED exists:', CLEANED_CANONICAL_PATH.exists())

ROOT: /Users/m2/Projects/STAT 5243/Project 1
RAW_PATH exists: True
CANONICAL CLEANED exists: True


## 2) Acquisition Integrity Checks

In [2]:
raw = pd.read_csv(RAW_PATH)

missing_cols = [c for c in REQUIRED_RAW_COLUMNS if c not in raw.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

print('Raw shape:', raw.shape)
print('Columns:', len(raw.columns))
print('Missing body % (raw):', round(raw['body'].isna().mean() * 100, 4))

Raw shape: (53187, 8)
Columns: 8
Missing body % (raw): 53.4886


## 3) Cleaning and Preprocessing

In [3]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http[s]?://\S+", "", text)
    text = re.sub(r"\[([^\]]+)\]\([^\)]+\)", r"\1", text)
    text = re.sub(r"[*_]{1,3}", "", text)
    text = re.sub(r"#{1,6}\s*", "", text)
    text = (
        text.replace("&amp;", "&")
        .replace("&lt;", "<")
        .replace("&gt;", ">")
        .replace("&nbsp;", " ")
        .replace("&#x200B;", "")
    )
    return re.sub(r"\s+", " ", text).strip()


def infer_post_type(url: str) -> str:
    if not isinstance(url, str):
        return "other"
    u = url.lower()
    if "v.redd.it" in u or "youtube" in u or "youtu.be" in u:
        return "video"
    if "i.redd.it" in u or u.endswith((".png", ".jpg", ".jpeg", ".gif")):
        return "image"
    if "reddit.com/r/wallstreetbets/comments" in u:
        return "text"
    return "link"


def preprocess(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()
    for col in ["score", "comms_num"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["score", "comms_num"]).copy()

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"]).copy()

    if "id" in df.columns:
        df = df.drop_duplicates(subset=["id"]).copy()
    else:
        df = df.drop_duplicates().copy()

    if "created" in df.columns:
        df = df.drop(columns=["created"])

    df["date"] = df["timestamp"].dt.date.astype(str)
    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.day_name()

    df["title_clean"] = df["title"].fillna("").map(clean_text)
    df["body_clean"] = df["body"].fillna("").map(clean_text)
    df["title_nlp"] = df["title_clean"].str.lower()

    df["has_body"] = df["body"].fillna("").str.strip().ne("")
    df["title_length"] = df["title_clean"].str.len()

    df["post_type"] = df["url"].map(infer_post_type)
    freq = df["post_type"].value_counts(normalize=True)
    keep = set(freq[freq >= 0.05].index)
    df["post_type_lumped"] = df["post_type"].where(df["post_type"].isin(keep), "Other")

    df["score_log"] = np.log1p(df["score"].clip(lower=0))
    df["comms_num_log"] = np.log1p(df["comms_num"].clip(lower=0))

    for col in ["score_log", "comms_num_log", "title_length", "hour"]:
        mu = df[col].mean()
        sd = df[col].std(ddof=0)
        mn = df[col].min()
        mx = df[col].max()
        df[f"{col}_zscore"] = (df[col] - mu) / (sd if sd else 1.0)
        df[f"{col}_minmax"] = (df[col] - mn) / ((mx - mn) if (mx - mn) else 1.0)

    df["day_of_week_encoded"] = df["day_of_week"].astype("category").cat.codes
    df["type_image"] = df["post_type_lumped"].eq("image")
    df["type_text"] = df["post_type_lumped"].eq("text")
    return df

In [4]:
cleaned = preprocess(raw)
print('Cleaned shape:', cleaned.shape)
print('Missing body % (cleaned):', round(cleaned['body'].isna().mean() * 100, 4))

cleaned_out = OUT_DIR / 'reddit_wsb_cleaned_full_workflow_code_notebook.csv'
cleaned.to_csv(cleaned_out, index=False)
print('Wrote:', cleaned_out)

Cleaned shape: (53187, 30)
Missing body % (cleaned): 53.4886
Wrote: /Users/m2/Projects/STAT 5243/Project 1/Project Workspace/Supporting Materials/Generated Outputs/one_code_notebook/reddit_wsb_cleaned_full_workflow_code_notebook.csv


## 4) EDA and Advanced Diagnostics

In [5]:
def save_dist(df: pd.DataFrame, col: str, stem: str, bins: int = 50):
    plt.figure(figsize=(8, 4))
    plt.hist(df[col].dropna(), bins=bins)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig(FIG_DIR / stem, dpi=220)
    plt.close()


def run_eda(df: pd.DataFrame) -> dict:
    save_dist(df, 'score', '03_eda_fig_01_score_distribution.png')
    save_dist(df, 'comms_num', '03_eda_fig_02_comms_distribution.png')
    save_dist(df, 'score_log', '03_eda_fig_03_score_log_distribution.png')
    save_dist(df, 'comms_num_log', '03_eda_fig_04_comms_log_distribution.png')
    save_dist(df, 'title_length', '03_eda_fig_05_title_length_distribution.png')
    save_dist(df, 'hour', '03_eda_fig_06_hour_distribution.png', bins=24)

    plt.figure(figsize=(7, 5))
    plt.scatter(df['score_log'], df['comms_num_log'], s=4, alpha=0.25)
    plt.xlabel('score_log')
    plt.ylabel('comms_num_log')
    plt.title('score_log vs comms_num_log')
    plt.tight_layout()
    plt.savefig(FIG_DIR / '03_eda_fig_10_score_comments_scatter.png', dpi=220)
    plt.close()

    dow = df.groupby('day_of_week', observed=False)['score_log'].mean().reindex([
        'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'
    ])
    plt.figure(figsize=(8, 4))
    dow.plot(kind='bar')
    plt.title('Mean score_log by day_of_week')
    plt.ylabel('Mean score_log')
    plt.tight_layout()
    plt.savefig(FIG_DIR / '03_eda_fig_14_score_dow_trend.png', dpi=220)
    plt.close()

    rho, pval = spearmanr(df['score_log'], df['comms_num_log'], nan_policy='omit')

    groups = [g['score_log'].dropna().values for _, g in df.groupby('post_type_lumped') if len(g) > 0]
    if len(groups) > 1:
        stat, p_kruskal = kruskal(*groups)
    else:
        stat, p_kruskal = float('nan'), float('nan')

    outlier_rate = float((df['score_log_zscore'].abs() >= 3).mean())

    return {
        'spearman_score_vs_comments': {'rho': float(rho), 'p_value': float(pval)},
        'kruskal_score_by_post_type': {'statistic': float(stat), 'p_value': float(p_kruskal)},
        'score_log_outlier_rate_abs_ge_3': outlier_rate,
        'rows': int(df.shape[0]),
        'columns': int(df.shape[1]),
    }

In [6]:
eda_stats = run_eda(cleaned)
(JSON_DIR / '03_eda_advanced_stats.json').write_text(json.dumps(eda_stats, indent=2))
print(json.dumps(eda_stats, indent=2))

{
  "spearman_score_vs_comments": {
    "rho": 0.785220719966194,
    "p_value": 0.0
  },
  "kruskal_score_by_post_type": {
    "statistic": 7549.467232814123,
    "p_value": 0.0
  },
  "score_log_outlier_rate_abs_ge_3": 0.0017297459905616034,
  "rows": 53187,
  "columns": 30
}


In [7]:
## --- Advanced EDA: Hypothesis Tests, Effect Sizes, Advanced Visualizations ---

# 1) Spearman correlation matrix with p-values
key_vars = ['score_log', 'comms_num_log', 'title_length', 'hour']
key_data = cleaned[key_vars].dropna()
rho_matrix, p_matrix = spearmanr(key_data)
rho_df = pd.DataFrame(rho_matrix, index=key_vars, columns=key_vars)
p_df = pd.DataFrame(p_matrix, index=key_vars, columns=key_vars)

plt.figure(figsize=(7, 6))
plt.imshow(rho_df.values, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(label='Spearman rho')
for i in range(len(key_vars)):
    for j in range(len(key_vars)):
        sig = '***' if p_df.iloc[i,j] < 0.001 else ('**' if p_df.iloc[i,j] < 0.01 else ('*' if p_df.iloc[i,j] < 0.05 else ''))
        plt.text(j, i, f'{rho_df.iloc[i,j]:.3f}{sig}', ha='center', va='center', fontsize=9)
plt.xticks(range(len(key_vars)), key_vars, rotation=45, ha='right')
plt.yticks(range(len(key_vars)), key_vars)
plt.title('Spearman Correlation Matrix (*** p<0.001, ** p<0.01, * p<0.05)')
plt.tight_layout()
plt.savefig(FIG_DIR / '03_eda_fig_20_spearman_matrix.png', dpi=220)
plt.close()
print('Saved: 03_eda_fig_20_spearman_matrix.png')

# 2) Chi-squared test: post_type_lumped vs viral status (top 5%)
viral_threshold = cleaned['score_log'].quantile(0.95)
cleaned['high_score'] = (cleaned['score_log'] >= viral_threshold).astype(int)
ct = pd.crosstab(cleaned['post_type_lumped'], cleaned['high_score'])
chi2, p_chi, dof, expected = chi2_contingency(ct)
n_obs = ct.values.sum()
min_dim = min(ct.shape[0], ct.shape[1]) - 1
cramers_v = np.sqrt(chi2 / (n_obs * min_dim)) if min_dim > 0 else 0.0
print(f'Chi-squared: chi2={chi2:.2f}, p={p_chi:.2e}, dof={dof}, Cramers V={cramers_v:.4f}')

# 3) Eta-squared from Kruskal-Wallis
groups_kw = [g['score_log'].dropna().values for _, g in cleaned.groupby('post_type_lumped') if len(g) > 0]
H_stat, p_kw = kruskal(*groups_kw)
k_groups = len(groups_kw)
n_total = sum(len(g) for g in groups_kw)
eta_squared = (H_stat - k_groups + 1) / (n_total - k_groups)
print(f'Kruskal-Wallis: H={H_stat:.2f}, p={p_kw:.2e}, eta_squared={eta_squared:.4f}')

# 4) Violin plots: score_log by post_type_lumped
types_sorted = sorted(cleaned['post_type_lumped'].unique())
data_by_type = [cleaned.loc[cleaned['post_type_lumped'] == t, 'score_log'].dropna().values for t in types_sorted]
fig, ax = plt.subplots(figsize=(10, 5))
parts = ax.violinplot(data_by_type, showmeans=True, showmedians=True)
ax.set_xticks(range(1, len(types_sorted) + 1))
ax.set_xticklabels(types_sorted, rotation=30, ha='right')
ax.set_ylabel('score_log')
ax.set_title('score_log Distribution by Post Type (Violin Plot)')
plt.tight_layout()
plt.savefig(FIG_DIR / '03_eda_fig_21_violin_by_type.png', dpi=220)
plt.close()
print('Saved: 03_eda_fig_21_violin_by_type.png')

# 5) Log-scale histogram of raw score (heavy tail)
plt.figure(figsize=(8, 4))
plt.hist(cleaned['score'].clip(lower=1), bins=100, log=True, edgecolor='black', linewidth=0.3)
plt.xlabel('Score')
plt.ylabel('Count (log scale)')
plt.title('Score Distribution (Log-Scale Y-Axis) -- Heavy Tail Visible')
plt.tight_layout()
plt.savefig(FIG_DIR / '03_eda_fig_22_score_logscale.png', dpi=220)
plt.close()
print('Saved: 03_eda_fig_22_score_logscale.png')

# 6) Hour x post_type interaction plot with 95% CI
interaction = cleaned.groupby(['hour', 'post_type_lumped'])['score_log'].agg(['mean', 'count', 'std']).reset_index()
interaction['se'] = interaction['std'] / np.sqrt(interaction['count'])
interaction['ci95'] = 1.96 * interaction['se']

fig, ax = plt.subplots(figsize=(12, 5))
for ptype in types_sorted:
    sub = interaction[interaction['post_type_lumped'] == ptype].sort_values('hour')
    ax.plot(sub['hour'], sub['mean'], label=ptype, marker='o', markersize=3)
    ax.fill_between(sub['hour'], sub['mean'] - sub['ci95'], sub['mean'] + sub['ci95'], alpha=0.15)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Mean score_log')
ax.set_title('Engagement by Hour x Post Type (95% CI bands)')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '03_eda_fig_23_hour_type_interaction.png', dpi=220)
plt.close()
print('Saved: 03_eda_fig_23_hour_type_interaction.png')

# 7) Temporal regime shift: pre-2021 vs Jan 2021 vs post-Jan 2021
cleaned['period'] = np.where(
    cleaned['timestamp'] < pd.Timestamp('2021-01-01'), 'pre_2021',
    np.where(cleaned['timestamp'] < pd.Timestamp('2021-02-01'), 'jan_2021_event', 'post_jan_2021')
)
period_stats = cleaned.groupby('period')['score_log'].agg(['mean', 'median', 'std', 'count'])
print('\nTemporal Regime Stats:')
print(period_stats)
period_groups = [g['score_log'].dropna().values for _, g in cleaned.groupby('period')]
H_period, p_period = kruskal(*period_groups)
print(f'KW across periods: H={H_period:.2f}, p={p_period:.2e}')

# 8) Word frequency: top 30 title words (stopwords removed)
stopwords = {'the','a','an','is','are','was','were','to','of','in','for','on','and','or','it','i',
             'my','this','that','with','at','by','from','be','as','so','if','not','but','do','no',
             'just','have','has','had','can','will','would','should','all','been','what','when',
             'how','why','who','am','get','got','we','they','you','your','its','our','he','she',
             'me','up','out','about','like','im','dont','its','ive','cant','one','much'}
all_tokens = cleaned['title_nlp'].dropna().str.findall(r'[a-z]{2,}').explode()
filtered = all_tokens[~all_tokens.isin(stopwords)]
top30 = filtered.value_counts().head(30)

plt.figure(figsize=(10, 5))
plt.bar(range(len(top30)), top30.values, color='steelblue')
plt.xticks(range(len(top30)), top30.index, rotation=60, ha='right', fontsize=8)
plt.ylabel('Frequency')
plt.title('Top 30 Title Words (Stopwords Removed)')
plt.tight_layout()
plt.savefig(FIG_DIR / '03_eda_fig_24_word_frequency.png', dpi=220)
plt.close()
print('Saved: 03_eda_fig_24_word_frequency.png')

# 9) Save extended EDA stats JSON
eda_advanced = {
    **eda_stats,
    'chi2_post_type_vs_viral': {'chi2': float(chi2), 'p_value': float(p_chi), 'dof': int(dof), 'cramers_v': float(cramers_v)},
    'kruskal_eta_squared': float(eta_squared),
    'temporal_regime': {
        'kruskal_H': float(H_period), 'p_value': float(p_period),
        'period_means': period_stats['mean'].to_dict(),
        'period_counts': period_stats['count'].to_dict(),
    },
    'top_10_words': top30.head(10).to_dict(),
}
(JSON_DIR / '03_eda_advanced_stats.json').write_text(json.dumps(eda_advanced, indent=2))
print('\nExtended EDA stats saved.')
print(f'  eta_squared = {eta_squared:.4f}')
print(f'  cramers_v   = {cramers_v:.4f}')
print(f'  regime H    = {H_period:.2f}')

Saved: 03_eda_fig_20_spearman_matrix.png
Chi-squared: chi2=3918.40, p=0.00e+00, dof=3, Cramers V=0.2714
Kruskal-Wallis: H=7549.47, p=0.00e+00, eta_squared=0.1419
Saved: 03_eda_fig_21_violin_by_type.png
Saved: 03_eda_fig_22_score_logscale.png
Saved: 03_eda_fig_23_hour_type_interaction.png

Temporal Regime Stats:
                    mean    median       std  count
period                                             
jan_2021_event  2.093000  0.693147  2.327175  19254
post_jan_2021   4.524168  4.442651  2.285856  33932
pre_2021        1.609438  1.609438       NaN      1
KW across periods: H=11818.57, p=0.00e+00
Saved: 03_eda_fig_24_word_frequency.png

Extended EDA stats saved.
  eta_squared = 0.1419
  cramers_v   = 0.2714
  regime H    = 11818.57


## 5) Feature Engineering, Diagnostics, and Ablation

In [8]:
POS_WORDS = {
    'gain', 'moon', 'bull', 'buy', 'rocket', 'win', 'green', 'profit', 'up', 'long', 'calls', 'squeeze'
}
NEG_WORDS = {
    'loss', 'bear', 'sell', 'drop', 'down', 'red', 'bagholder', 'panic', 'short', 'puts', 'crash'
}


def tokenize(text: str) -> list[str]:
    if not isinstance(text, str):
        return []
    return re.findall(r'[A-Za-z]{2,}', text.lower())


def simple_sentiment(text: str) -> float:
    toks = tokenize(text)
    if not toks:
        return 0.0
    pos = sum(t in POS_WORDS for t in toks)
    neg = sum(t in NEG_WORDS for t in toks)
    return (pos - neg) / len(toks)


def build_topic_features(df: pd.DataFrame, text_col: str) -> tuple[pd.DataFrame, dict]:
    tokenized = df[text_col].fillna('').astype(str).map(tokenize)
    dictionary = corpora.Dictionary(tokenized)
    dictionary.filter_extremes(no_below=30, no_above=0.5, keep_n=2000)
    corpus = [dictionary.doc2bow(tokens) for tokens in tokenized]

    if len(dictionary) == 0:
        topic_df = pd.DataFrame(np.zeros((len(df), 5)), columns=[f'topic_{i}' for i in range(5)])
        top_terms = {f'topic_{i}': [] for i in range(5)}
        return topic_df, top_terms

    lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=5, passes=3, random_state=SEED)
    topic_probs = []
    for bow in corpus:
        dist = lda.get_document_topics(bow, minimum_probability=0.0)
        dist_sorted = sorted(dist, key=lambda x: x[0])
        topic_probs.append([p for _, p in dist_sorted])

    topic_df = pd.DataFrame(topic_probs, columns=[f'topic_{i}' for i in range(5)])
    topic_df['dominant_topic'] = topic_df.values.argmax(axis=1)
    top_terms = {f'topic_{i}': [w for w, _ in lda.show_topic(i, topn=10)] for i in range(5)}
    return topic_df, top_terms

In [9]:
work = cleaned.copy()
text_col = 'title_nlp' if 'title_nlp' in work.columns else 'title_clean'
work['sentiment_score'] = work[text_col].fillna('').astype(str).map(simple_sentiment)

topic_df, top_terms = build_topic_features(work, text_col)
work = pd.concat([work.reset_index(drop=True), topic_df.reset_index(drop=True)], axis=1)

threshold = float(work['score'].quantile(0.95))
work['viral_flag'] = (work['score'] >= threshold).astype(int)
print(f'Virality threshold (score >= {threshold:.0f}), viral rate = {work["viral_flag"].mean():.4f}')

# --- Engineered pre-publication features (NO leakage: score/comms excluded) ---
# Temporal (cyclical encoding preserves periodicity)
work['hour_sin'] = np.sin(2 * np.pi * work['hour'] / 24)
work['hour_cos'] = np.cos(2 * np.pi * work['hour'] / 24)
work['late_night'] = ((work['hour'] >= 21) | (work['hour'] <= 1)).astype(int)

# Text intensity signals
work['caps_ratio'] = work['title'].str.count(r'[A-Z]') / work['title'].str.len().clip(lower=1)
work['exclaim_count'] = work['title'].str.count('!')
work['word_count'] = work['title_nlp'].str.split().str.len().fillna(0).astype(float)
work['unique_word_ratio'] = work['title_nlp'].str.split().apply(
    lambda x: len(set(x)) / max(len(x), 1) if isinstance(x, list) else 0.0)

# --- LEAKAGE-FREE feature sets (score_log, comms_num_log EXCLUDED) ---
base_cols = ['title_length', 'hour', 'has_body', 'type_image', 'type_text']
temporal_cols = ['hour_sin', 'hour_cos', 'late_night']
text_cols = ['caps_ratio', 'exclaim_count', 'word_count', 'unique_word_ratio']
topic_cols = [c for c in work.columns if c.startswith('topic_') and c != 'dominant_topic']

feature_sets = {
    'metadata_only':    base_cols,
    'plus_temporal':    base_cols + temporal_cols,
    'plus_text':        base_cols + temporal_cols + text_cols,
    'plus_sentiment':   base_cols + temporal_cols + text_cols + ['sentiment_score'],
    'full_model':       base_cols + temporal_cols + text_cols + ['sentiment_score'] + topic_cols,
}

sample = work.sample(min(20000, len(work)), random_state=SEED)
y = sample['viral_flag'].astype(int)
train_idx, test_idx = train_test_split(sample.index, test_size=0.25, random_state=SEED, stratify=y)
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

# --- 5-tier ablation ---
rows = []
for label, cols in feature_sets.items():
    cols_present = [c for c in cols if c in sample.columns]
    X = sample[cols_present].fillna(0.0).astype(float)
    model = LogisticRegression(max_iter=1200, class_weight='balanced', solver='liblinear')
    model.fit(X.loc[train_idx], y_train)
    pred_prob = model.predict_proba(X.loc[test_idx])[:, 1]
    pred = (pred_prob >= 0.5).astype(int)
    rows.append({
        'model': label,
        'feature_count': len(cols_present),
        'roc_auc': float(roc_auc_score(y_test, pred_prob)),
        'average_precision': float(average_precision_score(y_test, pred_prob)),
        'f1_at_0_5': float(f1_score(y_test, pred)),
    })

# Refit best (full) model for downstream diagnostics
best_cols = [c for c in feature_sets['full_model'] if c in sample.columns]
X_best = sample[best_cols].fillna(0.0).astype(float)
model_best = LogisticRegression(max_iter=1200, class_weight='balanced', solver='liblinear')
model_best.fit(X_best.loc[train_idx], y_train)
pred_prob = model_best.predict_proba(X_best.loc[test_idx])[:, 1]
pred = (pred_prob >= 0.5).astype(int)

# ROC curve (annotated)
fpr, tpr, _ = roc_curve(y_test, pred_prob)
auc_val = roc_auc_score(y_test, pred_prob)

# PR curve (annotated)
prec_arr, rec_arr, _ = precision_recall_curve(y_test, pred_prob)
ap_val = average_precision_score(y_test, pred_prob)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(fpr, tpr, label=f'ROC AUC = {auc_val:.4f}')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('ROC Curve'); ax1.legend()
ax2.plot(rec_arr, prec_arr, label=f'Avg Precision = {ap_val:.4f}')
ax2.axhline(y=y_test.mean(), color='k', linestyle='--', alpha=0.4, label=f'Baseline = {y_test.mean():.3f}')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve'); ax2.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '04_feature_fig_10_roc_pr_annotated.png', dpi=220)
plt.close()

# Confusion matrix
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap='Blues'); plt.colorbar()
plt.title('Confusion Matrix (Full Model)')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, int(cm[i, j]), ha='center', va='center')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(FIG_DIR / '04_feature_fig_06_confusion_matrix.png', dpi=220)
plt.close()

# Sentiment distribution
plt.figure(figsize=(8, 4))
plt.hist(work['sentiment_score'], bins=50)
plt.title('Sentiment Score Distribution')
plt.xlabel('Sentiment'); plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIG_DIR / '04_feature_fig_01_sentiment_distribution.png', dpi=220)
plt.close()

print('\n--- Ablation Results (LEAKAGE-FREE) ---')
for r in rows:
    print(f"  {r['model']:20s}  features={r['feature_count']:2d}  AUC={r['roc_auc']:.4f}  AP={r['average_precision']:.4f}  F1={r['f1_at_0_5']:.4f}")
print(f'\nFull model: AUC={auc_val:.4f}, AP={ap_val:.4f}')

Virality threshold (score >= 4189), viral rate = 0.0500

--- Ablation Results (LEAKAGE-FREE) ---
  metadata_only         features= 5  AUC=0.7235  AP=0.1011  F1=0.1831
  plus_temporal         features= 8  AUC=0.7295  AP=0.0997  F1=0.1829
  plus_text             features=12  AUC=0.7320  AP=0.1003  F1=0.1808
  plus_sentiment        features=13  AUC=0.7324  AP=0.1003  F1=0.1819
  full_model            features=18  AUC=0.7259  AP=0.1037  F1=0.1770

Full model: AUC=0.7259, AP=0.1037


In [10]:
## --- Ablation Chart + Advanced FE Diagnostics ---

ablation_df = pd.DataFrame(rows)
labels = ablation_df['model'].tolist()
x = np.arange(len(labels))
width = 0.25

plt.figure(figsize=(12, 5))
plt.bar(x - width, ablation_df['roc_auc'], width=width, label='ROC-AUC')
plt.bar(x, ablation_df['average_precision'], width=width, label='Avg Precision')
plt.bar(x + width, ablation_df['f1_at_0_5'], width=width, label='F1@0.5')
plt.xticks(x, labels, rotation=15, ha='right')
plt.ylim(0.0, 1.05)
plt.title('5-Tier Feature Ablation (Leakage-Free)')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '05_feature_ablation_comparison.png', dpi=220)
plt.close()

# --- VIF Analysis ---
def compute_vif(X_df):
    vif_data = []
    for col in X_df.columns:
        others = [c for c in X_df.columns if c != col]
        if len(others) == 0:
            continue
        r2 = LinearRegression().fit(X_df[others], X_df[col]).score(X_df[others], X_df[col])
        vif_val = 1.0 / (1.0 - r2) if r2 < 1.0 else float('inf')
        vif_data.append({'feature': col, 'VIF': round(vif_val, 2)})
    return pd.DataFrame(vif_data).sort_values('VIF', ascending=False)

vif_sample = sample[best_cols].fillna(0.0).astype(float).sample(min(10000, len(sample)), random_state=SEED)
vif_df = compute_vif(vif_sample)
print('\n--- VIF Analysis ---')
print(vif_df.to_string(index=False))

# --- 5-Fold Stratified CV ---
def cv_with_ci(X, y_cv, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    aucs, aps, f1s = [], [], []
    for train_i, test_i in skf.split(X, y_cv):
        m = LogisticRegression(max_iter=1200, class_weight='balanced', solver='liblinear')
        m.fit(X.iloc[train_i], y_cv.iloc[train_i])
        pp = m.predict_proba(X.iloc[test_i])[:, 1]
        preds = (pp >= 0.5).astype(int)
        aucs.append(roc_auc_score(y_cv.iloc[test_i], pp))
        aps.append(average_precision_score(y_cv.iloc[test_i], pp))
        f1s.append(f1_score(y_cv.iloc[test_i], preds))
    return {'auc_mean': np.mean(aucs), 'auc_std': np.std(aucs),
            'ap_mean': np.mean(aps), 'ap_std': np.std(aps),
            'f1_mean': np.mean(f1s), 'f1_std': np.std(f1s)}

X_cv = sample[best_cols].fillna(0.0).astype(float)
y_cv = sample['viral_flag'].astype(int)
cv_results = cv_with_ci(X_cv, y_cv)
print(f'\n--- 5-Fold CV ---')
print(f"  AUC = {cv_results['auc_mean']:.4f} +/- {cv_results['auc_std']:.4f}")
print(f"  AP  = {cv_results['ap_mean']:.4f} +/- {cv_results['ap_std']:.4f}")
print(f"  F1  = {cv_results['f1_mean']:.4f} +/- {cv_results['f1_std']:.4f}")

# --- Permutation Feature Importance ---
X_train_best = X_best.loc[train_idx]
X_test_best = X_best.loc[test_idx]
perm_imp = permutation_importance(model_best, X_test_best, y_test, n_repeats=10,
                                  random_state=SEED, scoring='roc_auc')
imp_df = pd.DataFrame({
    'feature': best_cols,
    'importance_mean': perm_imp.importances_mean,
    'importance_std': perm_imp.importances_std,
}).sort_values('importance_mean', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(imp_df['feature'], imp_df['importance_mean'], xerr=imp_df['importance_std'], color='steelblue')
plt.xlabel('Mean AUC Decrease')
plt.title('Permutation Feature Importance (Leakage-Free Full Model)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_DIR / '04_feature_fig_09_permutation_importance.png', dpi=220)
plt.close()
print('\n--- Top 5 Features by Permutation Importance ---')
print(imp_df.head().to_string(index=False))

# --- 3-Model Comparison ---
models_compare = {
    'LogisticRegression': LogisticRegression(max_iter=1200, class_weight='balanced', solver='liblinear'),
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, max_depth=8),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=SEED, max_depth=4),
}
model_comp_rows = []
for name, mdl in models_compare.items():
    mdl.fit(X_train_best, y_train)
    pp = mdl.predict_proba(X_test_best)[:, 1]
    preds = (pp >= 0.5).astype(int)
    model_comp_rows.append({
        'model': name,
        'roc_auc': float(roc_auc_score(y_test, pp)),
        'avg_precision': float(average_precision_score(y_test, pp)),
        'f1': float(f1_score(y_test, preds)),
    })
mc_df = pd.DataFrame(model_comp_rows)
print('\n--- Model Comparison ---')
print(mc_df.to_string(index=False))

# --- Temporal Validation Split ---
work_sorted = work.sort_values('timestamp').reset_index(drop=True)
cutoff = pd.Timestamp('2021-01-01')
train_temporal = work_sorted[work_sorted['timestamp'] < cutoff]
test_temporal = work_sorted[work_sorted['timestamp'] >= cutoff]

temporal_ok = (
    len(train_temporal) > 100
    and len(test_temporal) > 100
    and train_temporal['viral_flag'].nunique() == 2
    and test_temporal['viral_flag'].nunique() == 2
)

if temporal_ok:
    X_train_t = train_temporal[best_cols].fillna(0.0).astype(float)
    y_train_t = train_temporal['viral_flag'].astype(int)
    X_test_t = test_temporal[best_cols].fillna(0.0).astype(float)
    y_test_t = test_temporal['viral_flag'].astype(int)
    model_temporal = LogisticRegression(max_iter=1200, class_weight='balanced', solver='liblinear')
    model_temporal.fit(X_train_t, y_train_t)
    pp_t = model_temporal.predict_proba(X_test_t)[:, 1]
    temporal_auc = roc_auc_score(y_test_t, pp_t)
    print(f'\n--- Temporal Validation ---')
    print(f'  Temporal AUC (train<2021, test>=2021): {temporal_auc:.4f}')
    print(f'  Random-split CV AUC:                   {cv_results["auc_mean"]:.4f}')
    print(f'  Generalization gap:                    {cv_results["auc_mean"] - temporal_auc:.4f}')
else:
    # Fallback: use median date as split point to guarantee both classes in each split
    median_date = work_sorted['timestamp'].median()
    train_temporal = work_sorted[work_sorted['timestamp'] < median_date]
    test_temporal = work_sorted[work_sorted['timestamp'] >= median_date]
    X_train_t = train_temporal[best_cols].fillna(0.0).astype(float)
    y_train_t = train_temporal['viral_flag'].astype(int)
    X_test_t = test_temporal[best_cols].fillna(0.0).astype(float)
    y_test_t = test_temporal['viral_flag'].astype(int)
    if y_train_t.nunique() == 2 and y_test_t.nunique() == 2:
        model_temporal = LogisticRegression(max_iter=1200, class_weight='balanced', solver='liblinear')
        model_temporal.fit(X_train_t, y_train_t)
        pp_t = model_temporal.predict_proba(X_test_t)[:, 1]
        temporal_auc = roc_auc_score(y_test_t, pp_t)
        print(f'\n--- Temporal Validation (median-date split: {median_date.date()}) ---')
        print(f'  Temporal AUC (train<median, test>=median): {temporal_auc:.4f}')
        print(f'  Random-split CV AUC:                        {cv_results["auc_mean"]:.4f}')
        print(f'  Generalization gap:                         {cv_results["auc_mean"] - temporal_auc:.4f}')
    else:
        temporal_auc = float('nan')
        print('\n--- Temporal Validation: insufficient class diversity in splits ---')

# --- Save all FE diagnostics ---
ablation_payload = {
    'generated_at': datetime.now().isoformat(),
    'seed': SEED,
    'virality_threshold': threshold,
    'train_size': int(len(train_idx)),
    'test_size': int(len(test_idx)),
    'models': rows,
    'cv_results': cv_results,
    'model_comparison': model_comp_rows,
    'temporal_auc': float(temporal_auc) if not np.isnan(temporal_auc) else None,
    'vif': vif_df.to_dict(orient='records'),
    'top_permutation_features': imp_df.head(5).to_dict(orient='records'),
    'figure': str(FIG_DIR / '05_feature_ablation_comparison.png'),
}
(JSON_DIR / '05_feature_ablation_table.json').write_text(json.dumps(ablation_payload, indent=2, default=str))

feature_summary = {
    'roc_auc': float(auc_val),
    'average_precision': float(ap_val),
    'f1_at_0_5': float(f1_score(y_test, pred)),
    'virality_threshold': threshold,
    'topic_top_terms': top_terms,
}
(JSON_DIR / '04_feature_summary_metrics.json').write_text(json.dumps(feature_summary, indent=2))

print('\n--- All FE diagnostics saved ---')
print(f'F1 lift (full vs metadata-only): {rows[-1]["f1_at_0_5"] - rows[0]["f1_at_0_5"]:.4f}')


--- VIF Analysis ---
          feature          VIF
          topic_1 6.772330e+13
          topic_0 6.127346e+13
          topic_2 5.700759e+13
          topic_4 5.393532e+13
          topic_3 5.267368e+13
       word_count 2.623000e+01
     title_length 2.540000e+01
        type_text 6.680000e+00
         has_body 5.160000e+00
             hour 2.420000e+00
       type_image 2.370000e+00
         hour_sin 2.170000e+00
         hour_cos 2.050000e+00
       late_night 1.850000e+00
unique_word_ratio 1.260000e+00
       caps_ratio 1.130000e+00
    exclaim_count 1.060000e+00
  sentiment_score 1.050000e+00

--- 5-Fold CV ---
  AUC = 0.7349 +/- 0.0158
  AP  = 0.1175 +/- 0.0082
  F1  = 0.1757 +/- 0.0106

--- Top 5 Features by Permutation Importance ---
   feature  importance_mean  importance_std
 type_text         0.235159        0.013910
  hour_cos         0.030594        0.005296
type_image         0.028953        0.006830
      hour         0.024732        0.006103
  has_body         0.0

### Hypothesis-to-Feature Mapping

Each engineered feature group was motivated by a specific EDA finding:

| EDA Finding | Feature Group | Rationale | Expected Direction |
|---|---|---|---|
| Late-evening posting peaks (hours 20-23) correlate with higher engagement | `hour_sin`, `hour_cos`, `late_night` | Cyclical encoding preserves periodic structure; binary flag captures peak window | Positive for late_night |
| Image posts have significantly higher median engagement than text posts (KW p < 0.001) | `type_image`, `type_text` | One-hot encoding of dominant post types captures format effect | Image positive, text negative |
| Title length and body presence vary across engagement tiers | `title_length`, `has_body`, `word_count` | Structural signals about post effort and content richness | Mixed (body presence may be negative for meme-driven virality) |
| Heavy use of capitalization and exclamation in high-engagement titles (WSB culture) | `caps_ratio`, `exclaim_count`, `unique_word_ratio` | Intensity signals capturing emphatic or attention-seeking language | Positive for caps and exclamation |
| LDA topics capture thematic shifts (e.g., GME/AMC focus during meme-stock event) | `topic_0` ... `topic_4` | Probabilistic topic assignments reflect community attention focus | Mixed (topic-dependent) |
| Domain-specific sentiment lexicon (gain/moon/bull vs loss/bear/crash) | `sentiment_score` | WSB-specific polarity ratio; captures directional conviction in titles | Weak positive (limited by sarcasm) |

**Note on precision**: The model operates on a 5% base-rate virality task. Low precision at the 0.5 threshold is expected -- the PR curve should be consulted for threshold selection in practice.

## 6) Reproducibility and Export Manifest

In [11]:
manifest = {
    'generated_at': datetime.now().isoformat(),
    'raw_path': str(RAW_PATH),
    'canonical_cleaned_reference': str(CLEANED_CANONICAL_PATH),
    'outputs': {
        'cleaned_csv': str(cleaned_out),
        'json_dir': str(JSON_DIR),
        'fig_dir': str(FIG_DIR),
    },
    'eda_json': str(JSON_DIR / '03_eda_advanced_stats.json'),
    'ablation_json': str(JSON_DIR / '05_feature_ablation_table.json'),
}
(JSON_DIR / 'full_workflow_code_notebook_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

{
  "generated_at": "2026-02-19T10:20:40.785430",
  "raw_path": "/Users/m2/Projects/STAT 5243/Project 1/Project Deliverables/Datasets/reddit_wsb.csv",
  "canonical_cleaned_reference": "/Users/m2/Projects/STAT 5243/Project 1/Project Deliverables/Datasets/reddit_wsb_cleaned.csv",
  "outputs": {
    "cleaned_csv": "/Users/m2/Projects/STAT 5243/Project 1/Project Workspace/Supporting Materials/Generated Outputs/one_code_notebook/reddit_wsb_cleaned_full_workflow_code_notebook.csv",
    "json_dir": "/Users/m2/Projects/STAT 5243/Project 1/Project Workspace/Supporting Materials/Generated Outputs/one_code_notebook/artifacts/json",
    "fig_dir": "/Users/m2/Projects/STAT 5243/Project 1/Project Workspace/Supporting Materials/Generated Outputs/one_code_notebook/artifacts/figures"
  },
  "eda_json": "/Users/m2/Projects/STAT 5243/Project 1/Project Workspace/Supporting Materials/Generated Outputs/one_code_notebook/artifacts/json/03_eda_advanced_stats.json",
  "ablation_json": "/Users/m2/Projects/STAT 